A Dockerfile is a plain-text recipe that tells Docker how to build an image. Every image you use — `nginx:alpine`, `python:3.12-slim`, `node:20` — was built from a Dockerfile. Writing your own means your app can be packaged, shipped, and run anywhere Docker runs, with zero manual setup.

In this notebook we'll cover:
- The build context and how `docker build` works
- Every instruction you'll use regularly: `FROM`, `RUN`, `COPY`, `ADD`, `WORKDIR`, `ENV`, `ARG`, `EXPOSE`, `CMD`, `ENTRYPOINT`
- The difference between `CMD` and `ENTRYPOINT` — the most commonly confused pair
- Shell form vs exec form
- A complete worked example: packaging a Python web app

## How `docker build` Works

```bash
docker build [OPTIONS] PATH
```

When you run `docker build .`, Docker does two things:

1. **Sends the build context** — the directory you specify (`.` = current dir) is tarred up and sent to the Docker daemon. The daemon can only access files within this context.
2. **Executes each instruction** — the daemon reads the Dockerfile top to bottom, executing each instruction. Every instruction that changes the filesystem produces a new **layer** on top of the previous one.

```
docker build -t myapp:1.0 .
               │            │
               │            └── build context (current directory)
               └── tag: name:tag for the resulting image
```

### The Layer Cache

Docker caches each layer. If an instruction and all instructions before it are unchanged, Docker reuses the cached layer instead of re-running it. This makes rebuilds fast — as long as you order your instructions correctly (more on this in notebook 06).

A cache miss at layer N invalidates all layers after N — even if those instructions didn't change.

## `.dockerignore`

Before looking at instructions, there's one file that affects every build: `.dockerignore`.

It works exactly like `.gitignore` — list paths you don't want sent in the build context:

```
# .dockerignore
__pycache__/
*.pyc
.git/
.env
*.log
node_modules/
tests/
```

Why it matters:
- **Speed** — a smaller context is sent faster to the daemon
- **Security** — secrets in `.env` or `.git/` won't accidentally end up in the image
- **Cache stability** — files that change frequently (logs, test output) won't invalidate the layer cache

## `FROM` — Choose Your Base Image

Every Dockerfile starts with `FROM`. It sets the base image all subsequent instructions build on.

```dockerfile
FROM python:3.12-slim
```

### Choosing a base image

| Tag pattern | What it means |
|-------------|---------------|
| `python:3.12` | Full Debian — largest, most compatible |
| `python:3.12-slim` | Debian with non-essential packages removed |
| `python:3.12-alpine` | Alpine Linux — smallest, but uses musl libc (may cause compatibility issues) |
| `python:3.12-bookworm` | Debian Bookworm (explicit OS version) |

**Prefer `-slim` variants** for most production images — good balance of size and compatibility.

**Pin the exact tag** in production — `python:3.12.3-slim-bookworm` rather than `python:3.12-slim` to ensure builds are reproducible even if the upstream image is updated.

### `FROM scratch`

A special base meaning "start with an empty filesystem". Used for statically compiled binaries (Go, Rust) that don't need a base OS at all — produces the smallest possible images.

```dockerfile
FROM scratch
COPY myapp /myapp
CMD ["/myapp"]
```

## `RUN` — Execute Commands During Build

`RUN` executes a shell command and commits the result as a new layer.

```dockerfile
RUN apt-get update && apt-get install -y curl
```

### Chain commands in a single `RUN`

Each `RUN` is a layer. Running `apt-get update` in one `RUN` and `apt-get install` in another is an anti-pattern — the update result is cached separately and goes stale. Always combine update + install + cleanup in one `RUN`:

```dockerfile
RUN apt-get update \
 && apt-get install -y --no-install-recommends curl ca-certificates \
 && rm -rf /var/lib/apt/lists/*
```

The `rm -rf /var/lib/apt/lists/*` at the end deletes the package index cache — it's no longer needed and adds to the image size if left in.

### Shell form vs exec form

Every instruction that runs a process (`RUN`, `CMD`, `ENTRYPOINT`) has two forms:

```dockerfile
# Shell form — runs via /bin/sh -c
RUN echo "hello"

# Exec form — runs directly, no shell involved
RUN ["echo", "hello"]
```

Shell form is convenient because you get `&&`, `$VAR`, pipes, and other shell features. Exec form is preferred for `CMD` and `ENTRYPOINT` because it guarantees your process is PID 1 and receives signals correctly (SIGTERM on `docker stop`).

## `COPY` and `ADD` — Get Files Into the Image

### `COPY` — the right default

Copies files or directories from the **build context** into the image:

```dockerfile
COPY requirements.txt .          # copy one file to WORKDIR
COPY src/ /app/src/              # copy a directory
COPY *.py /app/                  # glob patterns work
COPY --chown=appuser:appuser . . # copy and set ownership
```

### `ADD` — only when you need its extras

`ADD` does everything `COPY` does, plus:
- Automatically **extracts** `.tar`, `.tar.gz`, `.tgz`, `.bz2`, `.xz` archives
- Accepts **URLs** as source (downloads during build)

```dockerfile
ADD myapp.tar.gz /opt/myapp/     # extracts automatically
ADD https://example.com/file.sh /tmp/file.sh  # download
```

**Rule of thumb:** Use `COPY` unless you specifically need archive extraction or URL fetching. `COPY` is explicit and predictable; `ADD` has side effects that surprise people.

For URL downloads, prefer `RUN curl` or `RUN wget` — it's clearer and you can chain cleanup in the same layer.

## `WORKDIR` — Set the Working Directory

`WORKDIR` sets the current directory for all subsequent `RUN`, `COPY`, `ADD`, `CMD`, and `ENTRYPOINT` instructions — and for the container's initial working directory at runtime.

```dockerfile
WORKDIR /app
COPY . .          # copies into /app
RUN pip install . # runs from /app
```

If the directory doesn't exist, `WORKDIR` creates it. You can set it multiple times — each call changes the active directory.

**Always use `WORKDIR` instead of `RUN cd /some/path`** — `cd` in a `RUN` instruction has no effect on subsequent instructions because each `RUN` starts from the `WORKDIR`.

## `ENV` and `ARG` — Variables

### `ENV` — runtime environment variables

`ENV` sets environment variables that are baked into the image and available at both build time and container runtime:

```dockerfile
ENV APP_ENV=production \
    LOG_LEVEL=info \
    PORT=8000
```

You can override `ENV` values at `docker run` time with `-e`. They are visible in `docker inspect`.

### `ARG` — build-time variables only

`ARG` declares a variable that only exists during the build — it is **not** present in the final image or at runtime:

```dockerfile
ARG PYTHON_VERSION=3.12
FROM python:${PYTHON_VERSION}-slim

ARG BUILD_DATE
ARG GIT_SHA
LABEL build.date=${BUILD_DATE} build.sha=${GIT_SHA}
```

Pass `ARG` values with `--build-arg`:

```bash
docker build --build-arg GIT_SHA=$(git rev-parse --short HEAD) -t myapp .
```

**Security note:** Don't use `ARG` for secrets — the values appear in the image's build history (`docker history`).

## `EXPOSE` — Document the Container's Port

`EXPOSE` declares which port the containerised application listens on:

```dockerfile
EXPOSE 8000
EXPOSE 8000/udp
```

**`EXPOSE` does not publish the port.** It is documentation — a signal to anyone running the image that they should map this port. You still need `-p 8000:8000` in `docker run` to actually bind it on the host.

`docker run -P` (uppercase P) maps all `EXPOSE`d ports to random host ports — this is where `EXPOSE` has a functional effect.

## `CMD` vs `ENTRYPOINT` — The Default Process

This is the most commonly confused part of Dockerfiles.

### `CMD` — default command, easily overridden

`CMD` sets the default command run when the container starts. It is completely replaced if you supply a command in `docker run`:

```dockerfile
CMD ["python", "app.py"]
```

```bash
docker run myapp           # runs: python app.py
docker run myapp bash      # runs: bash  (CMD is ignored)
```

### `ENTRYPOINT` — fixed executable, arguments are appended

`ENTRYPOINT` sets an executable that always runs. Arguments from `docker run` are **appended** to it rather than replacing it:

```dockerfile
ENTRYPOINT ["python"]
```

```bash
docker run myapp app.py    # runs: python app.py
docker run myapp --version # runs: python --version
```

### Using both together — the standard pattern

Use `ENTRYPOINT` for the fixed executable and `CMD` for default arguments that can be overridden:

```dockerfile
ENTRYPOINT ["python"]
CMD ["app.py"]
```

```bash
docker run myapp           # runs: python app.py   (CMD provides the default arg)
docker run myapp other.py  # runs: python other.py (CMD replaced by "other.py")
docker run --entrypoint bash myapp  # override ENTRYPOINT too
```

### Summary table

| | `ENTRYPOINT` | `CMD` |
|---|---|---|
| **Purpose** | The executable that always runs | Default arguments (or default command if no ENTRYPOINT) |
| **Overridden by** | `docker run --entrypoint` | Any command/args after the image name in `docker run` |
| **Form** | Exec form preferred | Exec form preferred |
| **Typical use** | Commands/tools with a fixed binary | Services (`["gunicorn", "app:create_app()"]`) |

> **Always use exec form** for `CMD` and `ENTRYPOINT` — `["executable", "arg"]` not `executable arg`. Shell form wraps the command in `/bin/sh -c`, making `sh` PID 1. That means `docker stop` sends SIGTERM to `sh`, not your app — graceful shutdown breaks.

## `LABEL` — Image Metadata

Key-value metadata attached to the image, readable with `docker inspect`:

```dockerfile
LABEL maintainer="team@example.com" \
      version="1.0.0" \
      description="My Python web service"
```

Labels don't affect runtime behaviour — they're useful for tooling, auditing, and filtering (`docker images --filter label=version=1.0.0`).

## Worked Example: Packaging a Python Web App

Here's a complete, realistic Dockerfile for a Python Flask application, with each instruction explained:

In [ ]:
dockerfile_content = '''\
# 1. Base image — slim Debian with Python 3.12 pre-installed
FROM python:3.12-slim

# 2. Metadata
LABEL maintainer="team@example.com" version="1.0.0"

# 3. Set environment variables
#    PYTHONDONTWRITEBYTECODE: no .pyc files in the image
#    PYTHONUNBUFFERED: stdout/stderr are not buffered (logs appear immediately)
ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1 \\
    PORT=8000

# 4. Set the working directory — all subsequent instructions run from here
WORKDIR /app

# 5. Install dependencies FIRST (before copying app code)
#    This layer is cached as long as requirements.txt doesn't change,
#    even if the application code changes — rebuilds stay fast.
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 6. Copy the rest of the application code
COPY . .

# 7. Document the port the app listens on
EXPOSE 8000

# 8. Default command — exec form, so gunicorn is PID 1 and receives signals
CMD ["gunicorn", "--bind", "0.0.0.0:8000", "app:create_app()"]
'''

print(dockerfile_content)

In [ ]:
import os, textwrap

# Create a minimal Flask app to build and run locally
os.makedirs("/tmp/myflaskapp", exist_ok=True)

# app.py
with open("/tmp/myflaskapp/app.py", "w") as f:
    f.write(textwrap.dedent("""\
        from flask import Flask

        def create_app():
            app = Flask(__name__)

            @app.route("/")
            def index():
                return "Hello from Docker!\\n"

            @app.route("/health")
            def health():
                return {"status": "ok"}

            return app
    """))

# requirements.txt
with open("/tmp/myflaskapp/requirements.txt", "w") as f:
    f.write("flask==3.0.3\ngunicorn==22.0.0\n")

# Dockerfile
with open("/tmp/myflaskapp/Dockerfile", "w") as f:
    f.write(textwrap.dedent("""\
        FROM python:3.12-slim
        ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
        WORKDIR /app
        COPY requirements.txt .
        RUN pip install --no-cache-dir -r requirements.txt
        COPY . .
        EXPOSE 8000
        CMD ["gunicorn", "--bind", "0.0.0.0:8000", "app:create_app()"]
    """))

# .dockerignore
with open("/tmp/myflaskapp/.dockerignore", "w") as f:
    f.write("__pycache__/\n*.pyc\n.env\n")

print("Files written to /tmp/myflaskapp/")
for name in sorted(os.listdir("/tmp/myflaskapp")):
    print(" ", name)

In [ ]:
# Build the image — watch the layer output to see caching in action
!docker build -t myflaskapp:1.0 /tmp/myflaskapp/

In [ ]:
# Inspect the resulting image — size, layers, metadata
!docker images myflaskapp
!echo
!docker inspect myflaskapp:1.0 --format 'Cmd: {{.Config.Cmd}}  Entrypoint: {{.Config.Entrypoint}}  WorkingDir: {{.Config.WorkingDir}}'

In [ ]:
# See every layer and what instruction produced it
!docker history myflaskapp:1.0

In [ ]:
# Run the app — detached, port-mapped, named
!docker run -d --name flask-demo -p 8000:8000 myflaskapp:1.0

import time; time.sleep(2)   # give gunicorn a moment to start

!curl -s http://localhost:8000/
!curl -s http://localhost:8000/health

In [ ]:
# Check gunicorn IS PID 1 (exec form CMD — no shell wrapper)
!docker exec flask-demo cat /proc/1/cmdline | tr '\0' ' '

In [ ]:
# Rebuild after a code change — requirements.txt unchanged, so pip layer is cached
with open("/tmp/myflaskapp/app.py", "a") as f:
    f.write("\n# a comment — simulates a code change\n")

!docker build -t myflaskapp:1.1 /tmp/myflaskapp/
# Notice: "CACHED" appears for the pip install layer — only the COPY . . layer reruns

In [ ]:
# Clean up
!docker rm -f flask-demo
!docker rmi myflaskapp:1.0 myflaskapp:1.1
!echo "Done"

## Dockerfile Instruction Quick Reference

| Instruction | Purpose | Notes |
|-------------|---------|-------|
| `FROM` | Set the base image | Always first; pin tags in production |
| `RUN` | Run a command during build | Combine with `&&`; clean up in same layer |
| `COPY` | Copy files from build context | Prefer over `ADD` for plain files |
| `ADD` | Copy + extract archives / fetch URLs | Use only when needed |
| `WORKDIR` | Set working directory | Creates the dir if it doesn't exist |
| `ENV` | Set runtime environment variables | Visible in `docker inspect` |
| `ARG` | Set build-time variables | Not in the final image or runtime |
| `EXPOSE` | Document the container's port | Does not publish the port |
| `CMD` | Default command/args | Replaced entirely by `docker run` args |
| `ENTRYPOINT` | Fixed executable | Args from `docker run` are appended |
| `LABEL` | Key-value metadata | For tooling and auditing only |

## Summary

- `docker build PATH` sends the build context to the daemon and executes each Dockerfile instruction as a layer. Use `.dockerignore` to exclude files that shouldn't be in the context.
- `FROM` sets the base — prefer `-slim` variants and pin exact tags for production.
- `RUN` runs commands during build. Chain update + install + cleanup in one `RUN` to keep layers small.
- `COPY` is the right default for getting files in. `ADD` adds archive extraction and URL fetching.
- `WORKDIR` sets the active directory — never use `RUN cd`.
- `ENV` bakes environment variables into the image (overridable at runtime). `ARG` exists only during the build.
- `EXPOSE` documents the port — it does not publish it.
- `CMD` provides the default command, replaced by `docker run` arguments. `ENTRYPOINT` sets a fixed executable that `docker run` arguments are appended to. Use both together for flexible, overridable defaults.
- Always use **exec form** (`["executable", "arg"]`) for `CMD` and `ENTRYPOINT` so your process is PID 1 and handles signals correctly.